**Author**: Felipe Matheus
**Purpose**: Experiment launcher for the annealing **tensile strength (UTS)** surrogate.

Same architecture as `run_experiments.ipynb` (IACS): all pipeline logic lives
in `src/modeling/Experiments.py`, which is process-agnostic — the SAME
`ExperimentRunner` is reused; only the `ExperimentConfig` changes (target,
features, physical bounds). No new class needed.

Results layout: `models/annealing_tensile_strength/experiments/`
(`experiments_log.csv` + one folder per run).

# 1. Setup

In [1]:
import logging
import os
import sys

import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from autogluon.tabular import TabularPredictor

module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.processing.Processing import Processing
from src.feature_engineering.FeatureEngineering import FeatureEngineering
from src.modeling.Modeling import Modeling
from src.metrics.Evaluation import Evaluation
from src.modeling.Experiments import ExperimentConfig, ExperimentRunner

from config.Variables import Variables

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

%load_ext autoreload
%autoreload 2

varv = Variables()
proc = Processing()
feng = FeatureEngineering()
modl = Modeling()
evla = Evaluation()
runr = ExperimentRunner(modl, evla, models_root=varv.PATHS.models)

c:\Users\fmfoa\Projects\uncertainty-aware-predictors\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TARGET = "tensile_strength_final"
ALL_FEATURES = ["purity", "initial_diameter", "tensile_strength", "temperature", "time"]
PROCESS = "annealing_uts"

SCHEMA_DATE = "110826"
FILE_NAME_SCHEMA_DATA = "schema_annealing_essays_{}.csv".format(SCHEMA_DATE)

TAG = f"annealing-schema{SCHEMA_DATE}"
GRID = {
    "time_limit_a": [900, 600],
    # "num_bag_folds_a": [10, 15, 20],
    # "features": [
    #     ("purity", "initial_diameter", "tensile_strength", "temperature", "time"),
    #     # ("initial_diameter", "tensile_strength", "temperature", "time"),
    # ],
}

# 2. Data (same preparation as annealing_uts.ipynb, run once)

In [3]:
df, df_val = proc.process_annealing_uts(
    features=ALL_FEATURES,
    target=TARGET,
    df_schema=pd.read_csv(os.path.join(varv.PATHS.data_raw, FILE_NAME_SCHEMA_DATA)),
)

In [4]:
# FILE_NAME = "dataset_annealing_tensile-strength.csv"

# df_raw = pd.read_csv(os.path.join(varv.PATHS.data_raw, FILE_NAME))
# df_float = proc.df_to_float(
#     df_raw, drop_cols=["DOI", "is_Cu"], ignore_columns=["material"]
# )
# df_labeled = feng.label_element(df_float).drop_duplicates()
# df_with_masks = feng.add_ratio_mask_column(
#     feng.add_ratio_mask_column(df_labeled, "grain_size"), "tensile_strength",
# )
# df = (
#     df_with_masks[ALL_FEATURES + [TARGET]]
#     .dropna(subset=[TARGET])
#     .reset_index(drop=True)
# )

# # No essay rows yet for UTS -> no is_essay column. The runr detects this
# # and applies uniform weights (weight_on_essay_rows must stay 1.0).
# print(f"Dataset: {df.shape}")

# # Validation set: none held-out yet. When UTS essays arrive, build df_val
# # from them (with the same columns) and pass df_val=df_val below.
# df_val = None
# df.head()

# 3. Base config

In [5]:
base = ExperimentConfig(
    process=PROCESS,
    tag=TAG,
    target=TARGET,
    features=tuple(ALL_FEATURES),
    y_max=None,
    y_min=0.0,
    presets_a = "best_quality",
)
print(base.run_id)

annealing-schema110826__0ac26579


# 4. Single run (sanity check before any grid)

Run the base config alone first; then repeat 2-3x with `tag="uts-v1-rep2"`
etc. to measure run-to-run noise (the floor below which grid differences
mean nothing).

In [6]:
result = runr.run_experiment(df, cfg=base, df_val=df_val)
result["artifacts"]["metrics"]

2026-08-11 15:33:33,396 | INFO | src.modeling.Experiments | === Running annealing-schema110826__0ac26579 ===
2026-08-11 15:33:33,398 | INFO | src.modeling.Experiments | No 'is_essay' column: dataset has no essay rows; uniform weights applied.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          8
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       5.84 GB / 31.57 GB (18.5%)
Disk Space Avail:   727.32 GB / 932.08 GB (78.0%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=5, num_bag_sets=1
Values in column 'weight_col' used as sample weights instead of predictive features. Evaluation metrics will ignore sample weights, specify weight_evaluation=Tr

{'rmse': 15.869155435262606,
 'mae': 9.792584643359376,
 'mape': 3.6048580428243397,
 'r2': 0.4720861193226151}

# 5 Validation set

TBD

In [7]:
df_val

,purity,initial_diameter,tensile_strength,temperature,time,tensile_strength_final
35,99.7637,2.0,349.58,723.0,60.0,239.570748
13,99.9000,1.2,397.09,573.0,60.0,262.400000
26,99.9740,0.8,347.70,823.0,0.5,301.290000
30,99.9740,0.8,347.70,873.0,20.0,242.340000
16,99.9000,1.2,397.09,523.0,60.0,260.260000
31,99.9740,0.8,347.70,873.0,30.0,239.190000
21,99.9800,1.2,335.21,523.0,30.0,245.680000
12,99.9000,1.2,397.09,573.0,30.0,261.320000
8,99.9000,1.2,431.58,623.0,30.0,239.410000
17,99.9000,1.2,397.09,523.0,90.0,259.140000


In [8]:
result["artifacts"]["validation_metrics"]

{'rmse': 13.076932872853806,
 'mae': 8.490315251275883,
 'mape': 3.1931300976824764,
 'r2': 0.4755122361089602,
 'coverage': {0.5: 0.7, 0.8: 0.9, 0.9: 0.9, 0.95: 1.0}}

In [9]:
result["artifacts"].keys()

dict_keys(['config', 'features', 'target', 'base_model_names', 'weights', 'nnls_recovery_ok', 'nnls_max_diff', 'variance_floor', 'recalibration_c', 'ood_ref', 'y_max', 'y_min', 'calibration_before', 'calibration_after', 'aleatoric_diagnostics', 'metrics', 'validation_metrics', 'dataset_hash'])

In [10]:
result.keys()

dict_keys(['cfg', 'run_dir', 'artifacts', 'log_row', 'predictor_a', 'predictor_b'])

In [11]:
result

{'cfg': ExperimentConfig(process='annealing_uts', tag='annealing-schema110826', base_tag=None, target='tensile_strength_final', features=('purity', 'initial_diameter', 'tensile_strength', 'temperature', 'time'), weight_on_essay_rows=1.0, presets_a='best_quality', num_bag_folds_a=5, num_bag_sets_a=1, num_stack_levels_a=0, time_limit_a=120, presets_b='medium_quality', num_bag_folds_b=5, num_stack_levels_b=0, time_limit_b=60, use_weighted_variance=True, variance_floor_frac=0.01, recalibration_target_alpha=0.9, calibration_alphas=(0.5, 0.8, 0.9, 0.95), y_max=None, y_min=0.0, fold_seed=42, use_shared_folds=False, group_col=None),
 'run_dir': Path('../../models/annealing_uts/experiments/annealing-schema110826/annealing-schema110826__0ac26579'),
 'artifacts': {'config': {'process': 'annealing_uts',
   'tag': 'annealing-schema110826',
   'base_tag': None,
   'target': 'tensile_strength_final',
   'features': ['purity',
    'initial_diameter',
    'tensile_strength',
    'temperature',
    'tim

# 6. Grid

In [12]:
log = runr.run_grid(df, base_cfg=base, grid=GRID, df_val=df_val)
log

2026-08-11 15:36:32,116 | INFO | src.modeling.Experiments | Grid: 2 runs over ['time_limit_a']
2026-08-11 15:36:32,118 | INFO | src.modeling.Experiments | === Running annealing-schema110826__time_limit_a=900__a30f4985 ===
2026-08-11 15:36:32,119 | INFO | src.modeling.Experiments | No 'is_essay' column: dataset has no essay rows; uniform weights applied.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          8
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       5.74 GB / 31.57 GB (18.2%)
Disk Space Avail:   716.56 GB / 932.08 GB (76.9%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=5, num_bag_sets=1
Values in column 'weight_col' used as sample we

,run_id,timestamp,elapsed_s,n_rows,dataset_hash,model_dir,cfg_process,cfg_tag,cfg_base_tag,cfg_target,...,mean_sigma_aleat,n_val,val_rmse,val_mae,val_mape,val_r2,val_cov_0.5,val_cov_0.8,val_cov_0.9,val_cov_0.95
0,annealing-schema110826__0ac26579,2026-08-11T15:36:30,176.7,36,57606bab,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_uts\experiments\annealing-schema110826\annealing-schema110826__0ac26579,annealing_uts,annealing-schema110826,NaN,tensile_strength_final,...,2.21508,10,13.07693,8.49032,3.19313,0.47551,0.7,0.9,0.9,1.0
1,annealing-schema110826__time_limit_a=900__a30f4985,2026-08-11T15:53:18,1006.0,36,57606bab,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_uts\experiments\annealing-schema110826\annealing-schema110826__time_limit_a=900__a30f4985,annealing_uts,annealing-schema110826__time_limit_a=900,annealing-schema110826,tensile_strength_final,...,2.21508,10,11.21791,7.49340,2.82315,0.61404,0.4,0.7,0.9,0.9
2,annealing-schema110826__time_limit_a=600__dce22b15,2026-08-11T16:04:57,699.1,36,57606bab,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_uts\experiments\annealing-schema110826\annealing-schema110826__time_limit_a=600__dce22b15,annealing_uts,annealing-schema110826__time_limit_a=600,annealing-schema110826,tensile_strength_final,...,2.21508,10,13.07693,8.49032,3.19313,0.47551,0.7,0.9,0.9,1.0


# 7. Inspect results

In [13]:
log = runr.load_log(base)

view_cols = [
    "run_id", "cfg_time_limit_a", "cfg_num_bag_folds_a", "cfg_features",
    "rmse", "mae", "cov_0.9", "val_rmse", "val_mae", "val_cov_0.9",
    "c_opt", "pct_truncated_aleat", "mean_sigma_epist", "mean_sigma_aleat",
    "elapsed_s",
]
log[[c for c in view_cols if c in log.columns]].sort_values("rmse")

,run_id,cfg_time_limit_a,cfg_num_bag_folds_a,cfg_features,rmse,mae,cov_0.9,val_rmse,val_mae,val_cov_0.9,c_opt,pct_truncated_aleat,mean_sigma_epist,mean_sigma_aleat,elapsed_s
1,annealing-schema110826__time_limit_a=900__a30f4985,900,5,purity|initial_diameter|tensile_strength|temperature|time,15.78935,9.70847,0.8889,11.21791,7.49340,0.9,1.4549,44.44,6.01499,2.21508,1006.0
0,annealing-schema110826__0ac26579,120,5,purity|initial_diameter|tensile_strength|temperature|time,15.86916,9.79258,0.9167,13.07693,8.49032,0.9,2.4038,38.89,5.46213,2.21508,176.7
2,annealing-schema110826__time_limit_a=600__dce22b15,600,5,purity|initial_diameter|tensile_strength|temperature|time,15.86916,9.79258,0.9167,13.07693,8.49032,0.9,2.4038,38.89,5.46213,2.21508,699.1


In [14]:
log.groupby("cfg_time_limit_a")[["rmse", "mae", "cov_0.9"]].agg(["mean", "std"])

rmse          mae     cov_0.9    
                      mean std     mean std    mean std
cfg_time_limit_a                                       
120               15.86916 NaN  9.79258 NaN  0.9167 NaN
600               15.86916 NaN  9.79258 NaN  0.9167 NaN
900               15.78935 NaN  9.70847 NaN  0.8889 NaN

# 8. Load a winner

In [15]:
best_row = log.sort_values("rmse").iloc[0]
RUN_ID = best_row["run_id"]
run_dir = Path(best_row["model_dir"])   # <-- absolute path, base_tag included

with open(run_dir / "artifacts.pkl", "rb") as f:
    art = pickle.load(f)
predictor_a = TabularPredictor.load(str(run_dir / "model_a"))
predictor_b = TabularPredictor.load(str(run_dir / "model_b"))

print(RUN_ID)
art["calibration_after"]

annealing-schema110826__time_limit_a=900__a30f4985


,alpha,empirical_coverage,gap
0,0.50,0.555556,0.055556
1,0.80,0.805556,0.005556
2,0.90,0.888889,-0.011111
3,0.95,0.888889,-0.061111


(raylet) The node with node id: daf9340905b585755bc1c4639f6fa8281d5eef8bd0d8e045c559877d and address: 127.0.0.1 and node name: 127.0.0.1 has been marked dead because the detector has missed too many heartbeats from it. This can happen when a 	(1) raylet crashes unexpectedly (OOM, etc.) 
	(2) raylet has lagging heartbeats due to slow network or busy workload.


In [16]:
1

1